# GenAI-Powered Digital Insights Assistant
### Grounded Marketing Analytics & Semantic Q&A System
**Role:** Senior Marketing Data Scientist  
**Core Objective:** Build a production-grade system that (1) automatically generates grounded executive campaign briefings, and (2) answers natural language marketing questions strictly citing verified pre-aggregated facts without hallucination.

## 1. Ground Truth Semantic Data Layer
We load the Bank Marketing dataset (41,188 contacts) and precompute 8 dimensional summary tables. This ensures the LLM never performs error-prone arithmetic on raw rows.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.data_layer import MarketingDataLayer

dl = MarketingDataLayer()
tables = dl.compute_all_summary_tables()
print(f"Computed {len(tables)} Ground Truth summary tables.")
tables['executive_overview']

## 2. Key Ground Truth Tables
Inspect channel performance and job segments.

In [ ]:
print('--- Performance by Contact Channel ---')
print(tables['campaign_by_channel'].to_string(index=False))

print('\n--- Top 5 Job Segments by Conversion Rate ---')
print(tables['campaign_by_job'].head(5).to_string(index=False))

## 3. Automated Executive Briefing Generation
Generates a structured executive briefing with automatic numerical grounding verification.

In [ ]:
from src.insight_generator import GroundedInsightGenerator

generator = GroundedInsightGenerator()
briefing_result = generator.generate_campaign_briefing(tables)

print(f"Provider: {briefing_result['provider']}")
print(f"Grounding Score: {briefing_result['verification']['grounding_score_pct']}%")
print("\n" + briefing_result['insight_report'])

## 4. Grounded Natural-Language Q&A (Lightweight Semantic RAG)
Queries are routed to the exact pre-aggregated ground truth tables, answered with strict constraints, and audited against the source numbers.

In [ ]:
from src.qa_engine import MarketingQAEngine

qa_engine = MarketingQAEngine(data_layer=dl)

queries = [
    'Which segment had the best conversion in the campaign?',
    'How did cellular compare to telephone in conversion rate and call duration?',
    'What was the conversion rate for students vs retirees?',
    'Which month had the lowest conversion rate and what was the volume in that month?',
    'How does a past successful campaign outcome impact conversion lift?',
    'What was the customer churn rate in 2025?' # Out of domain guardrail test
]

for q in queries:
    res = qa_engine.answer_question(q)
    print('='*70)
    print(f'QUESTION: {res["question"]}')
    print(f'RETRIEVED TABLES: {res["retrieved_tables"]}')
    print(f'ANSWER: {res["answer"]}')
    print(f'VERIFICATION: {res["verification"]["grounding_score_pct"]}% Grounded')


## 5. Grounding & Anti-Hallucination Benchmark Evaluation
Run the full benchmark evaluation suite.

In [ ]:
from src.evaluation import GroundingBenchmarkEvaluator

evaluator = GroundingBenchmarkEvaluator()
benchmark = evaluator.run_benchmark()
print(f"Retrieval Precision: {benchmark['retrieval_precision_pct']}%")
print(f"Grounded Fact Recall: {benchmark['grounded_fact_recall_pct']}%")
print(f"Out-of-Domain Guardrail Accuracy: {benchmark['out_of_domain_guardrail_accuracy_pct']}%")